# Data Cleaning — Focus Sessions Dataset

This notebook takes the raw google form responses
and transforms it into a cleaned dataset
ready for analysis and visualization.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Load the raw data

In [2]:
import pandas as pd
import numpy as np
import re
import math

RAW_PATH = "/content/drive/MyDrive/Colab Notebooks/Visual Analytics/form_responses.csv"
PROCESSED_PATH = "/content/drive/MyDrive/Colab Notebooks/Visual Analytics/cleaned_data.csv"

In [3]:
df_raw = pd.read_csv(RAW_PATH, keep_default_na=False, na_values=[""])
print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")
df_raw.head(3)

Loaded 80 rows, 20 columns


,Timestamp,What date did this study/work session take place?,What time did the session start?,What time did the session end?,What type of task were you doing?,How interesting was the task?,How difficult was the task?,How long did you plan to work/study? Enter the number of minutes. Example: 120,How many minutes after starting the session did you experience your first distraction? Enter a whole number. If no distraction occurred enter 0,How many distractions occurred? (Whole Number),How focused were you during the session?,Where did you study/work?,How would you describe the noise level during this session?,How many other people were around you during this session?,How much interaction did you have with other people during the session?,How many hours did you sleep the previous night? Example: 7.5,How many minutes of phone screen time had you accumulated before this session? Example: 145,What caffeine did you consume before the session?,Which focus strategies did you use,How effective was this study/work session overall? (Rate 1-5)
0,9/21/26 12:24,9/21/26,8:30:00 AM,11:00:00 AM,Studying,5,7,120,10,16,9,Office in Workplace,Quiet,1-2 people,None – no interaction,6,105,Coffee,Noise cancelling headphones,5- Extremely effective
1,9/21/26 12:41,9/21/26,10:00:00 AM,11:25:00 AM,Writing,5,7,90,3,19,5,Home,Very Quiet,3-5 people,Some,4.5,40,None,"Background Music, Noise cancelling headphones",3- Moderately effective
2,9/21/26 13:06,9/21/26,11:15:00 AM,12:40:00 PM,Reading,4,8,90 min,3 minutes,21,4,Home,Moderate,Alone,None – no interaction,9,NaN,Coffee,None,3- Moderately effective


## 2. Rename columns

The original Google Form question text makes unwieldy column names. We
rename them to short, snake_case names.

In [4]:
RENAME_MAP = {
    "Timestamp": "Timestamp",
    "What date did this study/work session take place?": "Date",
    "What time did the session start?": "Start_Time",
    "What time did the session end?": "End_Time",
    "What type of task were you doing?": "Task_Type",
    "How interesting was the task?": "Task_Interest",
    "How difficult was the task?": "Task_Difficulty",
    "How long did you plan to work/study? Enter the number of minutes. Example: 120": "Planned_Duration",
    "How many minutes after starting the session did you experience your first distraction? Enter a whole number. If no distraction occurred enter 0 ": "Time_to_First_Distraction",
    "How many distractions occurred? (Whole Number)": "Num_Distractions",
    "How focused were you during the session?": "Focus_Rating",
    "Where did you study/work?": "Location",
    "How would you describe the noise level during this session?": "Noise_Level",
    "How many other people were around you during this session?": "People_Present",
    "How much interaction did you have with other people during the session?": "Interaction_Level",
    "How many hours did you sleep the previous night? Example: 7.5": "Sleep_Hours",
    "How many minutes of phone screen time had you accumulated before this session? Example: 145": "Phone_Screen_Time",
    "What caffeine did you consume before the session?": "Caffeine",
    "Which focus strategies did you use": "Focus_Strategies",
    "How effective was this study/work session overall? (Rate 1-5)": "Effectiveness",
}

df = df_raw.rename(columns=RENAME_MAP)
df.columns.tolist()

['Timestamp',
 'Date',
 'Start_Time',
 'End_Time',
 'Task_Type',
 'Task_Interest',
 'Task_Difficulty',
 'Planned_Duration',
 'Time_to_First_Distraction',
 'Num_Distractions',
 'Focus_Rating',
 'Location',
 'Noise_Level',
 'People_Present',
 'Interaction_Level',
 'Sleep_Hours',
 'Phone_Screen_Time',
 'Caffeine',
 'Focus_Strategies',
 'Effectiveness']

In [5]:
df.isna().sum()

,0
Timestamp,0
Date,0
Start_Time,0
End_Time,0
Task_Type,0
Task_Interest,0
Task_Difficulty,0
Planned_Duration,0
Time_to_First_Distraction,0
Num_Distractions,0


### Inspecting the raw values

Before parsing anything, we look at rows with blank answers and at the
free-text numeric fields. Several of these are not missing, just written as
text (`"4 hours"`, `"300 minutes"`, `"Option 1"`). Those have to be parsed
**before** we decide what is actually missing.

In [6]:
# Find and display the rows with missing values in our renamed DataFrame `df`
columns_with_nas = ['Interaction_Level', 'Sleep_Hours', 'Phone_Screen_Time', 'Caffeine','Focus_Strategies']
rows_with_nas = df[df[columns_with_nas].isna().any(axis=1)]

print(f"Found {len(rows_with_nas)} rows with at least one missing value in: {columns_with_nas}")
display(rows_with_nas[columns_with_nas].head(10))

Found 25 rows with at least one missing value in: ['Interaction_Level', 'Sleep_Hours', 'Phone_Screen_Time', 'Caffeine', 'Focus_Strategies']


,Interaction_Level,Sleep_Hours,Phone_Screen_Time,Caffeine,Focus_Strategies
2,None – no interaction,9,NaN,Coffee,None
4,None – no interaction,6.5,NaN,None,Timer
6,NaN,240,Option 1,NaN,"Noise cancelling headphones, Timer"
7,NaN,4 hours,2 hours,NaN,NaN
8,Very little,300 minutes,240 minutes,Soda,NaN
10,NaN,480,30 minutes,NaN,NaN
12,Some,7,200,Coffee,NaN
13,A lot,240 minutes,248 minutes,NaN,"Background Music, Noise cancelling headphones,..."
14,A lot,480,180,NaN,NaN
15,A lot,6,8,NaN,NaN


## 3. Parsing messy numeric free-text fields

Several fields were collected as free text and came back inconsistently
formatted — plain numbers, `"90min"`, `"3 hours"`, `"Around 30"`, and in
one case the clearly-invalid text `"Option 1"` (a stray value that isn't
a time at all).

We write one small parser that:
- extracts the first number in the text,
- checks the remaining words against a short allow-list of recognized
  units (`hour`, `hours`, `hr`, `min`, `mins`, `minute`, `minutes`) and
  hedge words (`around`, `about`, `approx`, `roughly`),
- and returns a missing value if any *other*, unrecognized word is
  present — this is what correctly catches `"Option 1"` as invalid
  rather than misreading it as "1 minute".

This same parser, with a different output unit, is reused for:
`Planned_Duration`, `Time_to_First_Distraction`, `Sleep_Hours` (output in
hours), and `Phone_Screen_Time` (output in minutes).


In [7]:
TIME_UNIT_WORDS = {"hour", "hours", "hr", "hrs", "min", "mins", "minute", "minutes"}
HEDGE_WORDS = {"around", "about", "approx", "approximately", "roughly"}
ALLOWED_WORDS = TIME_UNIT_WORDS | HEDGE_WORDS

def extract_number_and_unit(val):
    """Return (number, is_hours) or (None, None) if the text isn't a
    usable numeric/time value (e.g. 'Option 1')."""
    if pd.isna(val):
        return None, None
    s = str(val).strip().lower()
    words = re.findall(r"[a-zA-Z]+", s)
    if not all(w in ALLOWED_WORDS for w in words):
        return None, None  # contains an unrecognized word -> garbage entry
    match = re.search(r"\d+\.?\d*", s)
    if not match:
        return None, None
    num = float(match.group())
    is_hours = bool({"hour", "hours", "hr", "hrs"} & set(words))
    return num, is_hours

def parse_minutes(val):
    """Parse free text into a number of minutes."""
    num, is_hours = extract_number_and_unit(val)
    if num is None:
        return np.nan
    return num * 60 if is_hours else num

def parse_hours(val):
    """Parse free text into a number of hours (used for sleep_hours,
    where a bare number > 16 is assumed to be minutes, not hours)."""
    num, is_hours = extract_number_and_unit(val)
    if num is None:
        return np.nan
    if is_hours:
        return round(num, 2)
    if num > 16:
        return round(num / 60, 2)
    return round(num, 2)

In [8]:
NUMERIC_TEXT_COLS = ["Planned_Duration", "Time_to_First_Distraction", "Sleep_Hours", "Phone_Screen_Time"]
_raw_numeric = df[NUMERIC_TEXT_COLS].copy()   # kept only to show the before/after below

df["Planned_Duration"] = df["Planned_Duration"].apply(parse_minutes)
df["Time_to_First_Distraction"] = df["Time_to_First_Distraction"].apply(parse_minutes)
df["Sleep_Hours"] = df["Sleep_Hours"].apply(parse_hours)
df["Phone_Screen_Time"] = df["Phone_Screen_Time"].apply(parse_minutes)

# Before/after for every value that was not already a plain number
_is_text = _raw_numeric.apply(lambda s: pd.to_numeric(s, errors="coerce").isna() & s.notna())
for col in NUMERIC_TEXT_COLS:
    changed = _is_text[col]
    if changed.any():
        print(f"\n{col}:")
        print(pd.DataFrame({"raw": _raw_numeric.loc[changed, col], "parsed": df.loc[changed, col]}).to_string())


Planned_Duration:
             raw  parsed
2         90 min    90.0
3          30min    30.0
4        1 hours    60.0
8     60 minutes    60.0
9          90min    90.0
10   120 minutes   120.0
11      30 mins     30.0
14  300 minutes    300.0
19       3 hours   180.0
21    30 minutes    30.0
22        60 min    60.0
24          3 hr   180.0
33       120 min   120.0
36        30 min    30.0
39       2 hours   120.0
40      120 mins   120.0
42          1 hr    60.0
52         90min    90.0
63       2 hours   120.0
75       150 min   150.0

Time_to_First_Distraction:
            raw  parsed
2     3 minutes     3.0
3      Around 4     4.0
8   20 minutes     20.0
9         30min    30.0
10   45 minutes    45.0
13    Around 30    30.0
19         5min     5.0
20     Around 3     3.0
29        10min    10.0
34    6 minutes     6.0
42    7 minutes     7.0
57        11min    11.0
58      about 5     5.0
59         9min     9.0

Sleep_Hours:
             raw  parsed
3        510 min     8.5
7   

## 4. Distraction count

`"How many distractions occurred?"` should be a whole number, but one
response reads `"20+"`. We strip non-numeric characters, and round any
decimal up to the next whole number (so partial/uncertain counts don't
get silently truncated down).

In [9]:
def parse_distraction_count(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    match = re.search(r"\d+\.?\d*", s)
    if not match:
        return np.nan
    return int(math.ceil(float(match.group())))

df["Num_Distractions"] = df["Num_Distractions"].apply(parse_distraction_count)
df[["Num_Distractions"]]

,Num_Distractions
0,16
1,19
2,21
3,5
4,1
...,...
75,28
76,36
77,28
78,12


## 5. Filling missing values

After parsing, a value is missing only if the respondent left it blank or
wrote something unusable (e.g. `"Option 1"`).

- **Numeric fields** are filled with the column **median**, not 0. A 0 would be a
  false value (e.g. 0 hours of sleep or 0 minutes of screen time). Each one gets
  an `_imputed` flag so analyses such as sleep vs. focus can exclude filled rows.
- **Categorical fields** left blank are filled with `"None"` (no caffeine, no
  strategy, no interaction), which is how a blank answer to those questions reads.
- Real zeros, such as 0 distractions, are left alone.

In [10]:
NUMERIC_FILL = ["Sleep_Hours", "Phone_Screen_Time", "Planned_Duration", "Time_to_First_Distraction"]

for col in NUMERIC_FILL:
    df[f"{col}_imputed"] = df[col].isna()
    median_val = df[col].median()
    print(f"{col}: filling {df[col].isna().sum()} values with median {median_val}")
    df[col] = df[col].fillna(median_val)

df["Interaction_Level"] = df["Interaction_Level"].fillna("None – no interaction")
df["Caffeine"] = df["Caffeine"].fillna("None")
df["Focus_Strategies"] = df["Focus_Strategies"].fillna("None")

for col in ["Task_Type", "Location", "Noise_Level", "People_Present"]:
    df[col] = df[col].fillna("Unknown")

print("\nTotal missing values remaining:", int(df.isna().sum().sum()))

Sleep_Hours: filling 4 values with median 6.5
Phone_Screen_Time: filling 8 values with median 175.0
Planned_Duration: filling 0 values with median 90.0
Time_to_First_Distraction: filling 0 values with median 6.0

Total missing values remaining: 0


## 6. Dates and times

**Data quality issue found:** one response has `date = 12/9/2000`, which
doesn't match the pattern of any other response and the
auto-logged form `Timestamp` for that same row is a normal 2026 date.

We use the auto-generated `Timestamp` as the session date.

Start/end times are then standardized to 24-hour `HH:MM:SS` format.

In [11]:
df["Date"] = pd.to_datetime(df["Timestamp"], format="%m/%d/%y %H:%M", errors="coerce").dt.date

_start_dt = pd.to_datetime(df["Start_Time"], format="%I:%M:%S %p", errors="coerce")
_end_dt = pd.to_datetime(df["End_Time"], format="%I:%M:%S %p", errors="coerce")

df["Start_Time"] = _start_dt.dt.strftime("%H:%M:%S")
df["End_Time"] = _end_dt.dt.strftime("%H:%M:%S")

print("Unparsed dates/times:", int(df["Date"].isna().sum()), int(_start_dt.isna().sum()), int(_end_dt.isna().sum()))
df[["Date", "Start_Time", "End_Time"]].head(5)

Unparsed dates/times: 0 0 0


,Date,Start_Time,End_Time
0,2026-09-21,08:30:00,11:00:00
1,2026-09-21,10:00:00,11:25:00
2,2026-09-21,11:15:00,12:40:00
3,2026-09-21,20:00:00,20:25:00
4,2026-09-22,21:00:00,09:25:00


## 7. Total session duration

Computed from start and end time. A few sessions have an end time earlier
than the start time (e.g. 4:20 PM → 4:58 AM) — these genuinely crossed
midnight, so we add 24 hours in that case rather than returning a
negative duration.

We also flag unusually long sessions (> 5 hours) for manual review. A
few of these (e.g. an 8:00 PM → 9:20 AM "session") look more like an
AM/PM entry mistake than a real 13-hour study session, but we don't
silently guess-correct them — we flag them so they can be checked
against the original form response by whoever submitted it.

In [12]:
_start_min = _start_dt.dt.hour * 60 + _start_dt.dt.minute
_end_min = _end_dt.dt.hour * 60 + _end_dt.dt.minute
_duration = _end_min - _start_min
_duration = _duration.where(_duration >= 0, _duration + 24 * 60)

df["Total_Duration_Minutes"] = _duration
df["Duration_Flag_Long"] = df["Total_Duration_Minutes"] > 300

n_flagged = df["Duration_Flag_Long"].sum()
print(f"Flagged {n_flagged} sessions with duration > 5 hours for review:")
df.loc[df["Duration_Flag_Long"], ["Date", "Start_Time", "End_Time", "Total_Duration_Minutes"]]

Flagged 6 sessions with duration > 5 hours for review:


,Date,Start_Time,End_Time,Total_Duration_Minutes
4,2026-09-22,21:00:00,09:25:00,745
12,2026-09-22,16:20:00,04:58:00,758
14,2026-09-22,12:30:00,18:00:00,330
40,2026-09-27,20:00:00,09:20:00,800
55,2026-09-30,13:30:00,02:50:00,800
64,2026-10-04,19:00:00,09:40:00,880


## 8. Standardize single-select categorical fields

For `task_type`, `location`, `noise_level`, and `caffeine`, we map each
response to the exact set of options offered on the form. Anything that
doesn't match an option exactly (e.g. `"Bus"`, `"Campus"`, `"Light lunch"`)
gets bucketed into `"Other"`, per the form's own design.

**One exception:** `"Meetings"` (plural) and `"Meeting"` are collapsed
into a single `"Meeting"` category — these are clearly the same answer
option, duplicated by a typo in the form itself, not two different task
types.

In [13]:
def map_to_allowed(val, allowed_set, other_label="Other"):
    if pd.isna(val):
        return np.nan
    v = str(val).strip()
    for opt in allowed_set:
        if v.lower() == opt.lower():
            return opt
    return other_label

TASK_TYPES = {"Studying", "Coding", "Reading", "Meeting", "Chores", "Writing",
              "Homework/Assignments", "Research", "Other"}

def map_task_type(val):
    if pd.isna(val):
        return np.nan
    v = str(val).strip()
    if v.lower() == "meetings":
        v = "Meeting"
    return map_to_allowed(v, TASK_TYPES)

LOCATIONS = {"Home", "Library", "Cafe", "Private Booth", "Other"}
NOISE_LEVELS = {"Very Quiet", "Quiet", "Moderate", "Noisy", "Very Noisy", "Other"}
CAFFEINE = {"Coffee", "Tea", "Energy drink", "Soda", "Don't remember", "None", "Other"}

df["Task_Type"] = df["Task_Type"].apply(map_task_type)
df["Location"] = df["Location"].apply(lambda v: map_to_allowed(v, LOCATIONS))
df["Noise_Level"] = df["Noise_Level"].apply(lambda v: map_to_allowed(v, NOISE_LEVELS))
df["Caffeine"] = df["Caffeine"].apply(lambda v: map_to_allowed(v, CAFFEINE))

df[["Task_Type", "Location", "Noise_Level", "Caffeine"]]

,Task_Type,Location,Noise_Level,Caffeine
0,Studying,Other,Quiet,Coffee
1,Writing,Home,Very Quiet,None
2,Reading,Home,Moderate,Coffee
3,Homework/Assignments,Library,Very Quiet,Coffee
4,Studying,Library,Quiet,None
...,...,...,...,...
75,Studying,Home,Very Quiet,Coffee
76,Studying,Library,Quiet,Tea
77,Homework/Assignments,Home,Quiet,None
78,Studying,Library,Very Quiet,Coffee


## 9. One-hot encode focus strategies (multi-select)

This question allows multiple answers in one cell (e.g.
`"Background Music, Noise cancelling headphones, Timer"`), so a single
category column won't work. We split each response on commas and create
one boolean column per form option. Any item that isn't one of the
listed options (e.g. `"Airpods"`) is counted under `strategy_other`,
consistent with how we handle unlisted answers elsewhere.

In [14]:
STRATEGIES = ["Phone in another room", "Focus app", "Background Music", "Noise cancelling headphones", "Timer", "None", "Other"]

def one_hot_strategies(val):
    cols = {f"strategy_{s.lower().replace(' ', '_')}": 0 for s in STRATEGIES}
    if pd.isna(val) or str(val).strip() == "":
        return cols
    for item in str(val).split(","):
        item = item.strip()
        matched = False
        for s in STRATEGIES:
            if item.lower() == s.lower():
                cols[f"strategy_{s.lower().replace(' ', '_')}"] = 1
                matched = True
                break
        if not matched:
            cols["strategy_other"] = 1
    return cols

_strategy_cols = df["Focus_Strategies"].apply(one_hot_strategies).apply(pd.Series)
df = pd.concat([df, _strategy_cols], axis=1)
df[["Focus_Strategies"] + list(_strategy_cols.columns)]

,Focus_Strategies,strategy_phone_in_another_room,strategy_focus_app,strategy_background_music,strategy_noise_cancelling_headphones,strategy_timer,strategy_none,strategy_other
0,Noise cancelling headphones,0,0,0,1,0,0,0
1,"Background Music, Noise cancelling headphones",0,0,1,1,0,0,0
2,None,0,0,0,0,0,1,0
3,None,0,0,0,0,0,1,0
4,Timer,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...
75,Lofi playlist,0,0,0,0,0,0,1
76,Background Music,0,0,1,0,0,0,0
77,Phone in another room,1,0,0,0,0,0,0
78,None,0,0,0,0,0,1,0


## 10. Extract numeric effectiveness rating

`Effectiveness` stores both a number and a label in one string (e.g.
`"3- Moderately effective"`). We keep just the leading number (1–5) for analysis.

In [15]:
df["Effectiveness"] = df["Effectiveness"].str.extract(r"(\d+)").astype(float)
df[["Effectiveness"]]

,Effectiveness
0,5.0
1,3.0
2,3.0
3,4.0
4,4.0
...,...
75,4.0
76,3.0
77,4.0
78,4.0


## 11. Save the cleaned data

In [17]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved {len(df)} rows x {len(df.columns)} columns to {PROCESSED_PATH}")

Saved 80 rows x 33 columns to /content/drive/MyDrive/Colab Notebooks/Visual Analytics/cleaned_data.csv
